## **Hiver SDE Intern Take-Home Assignment**

SupportSense: An AI-Powered Customer Support Agent Using Twitter Conversations
---------------------------------------------------------------------
**Objective**:
Explore the Customer Support on Twitter dataset, understand its structure, select one brand for analysis, and identify the most common customer support intents. This notebook performs initial data loading, preprocessing, exploratory data analysis (EDA), and conversation inspection, which will be used to build an AI support agent in the subsequent notebooks.

Description

SupportSense is an AI-powered customer support system built using real Twitter conversations. The system automatically classifies customer messages into predefined support intents, generates context-aware responses based on historical customer support interactions, and determines whether a customer issue should be automatically resolved or escalated to a human support agent.

Features
* Intent Classification
* AI Reply Generation
* Auto Handle / Human Escalation
* Evaluation Dashboard
* Golden Dataset
* Failure Analysis

**Step 1: Get the data, pick a brand, reconstruct threads**

In [1]:
!pip install -q kaggle

In [2]:
import os

# os.environ['KAGGLE_API_TOKEN']='KGAT_3dee322ff855d5f0f93956920e960557'

**Download and unzip Dataset**

In [3]:
!kaggle datasets download -d thoughtvector/customer-support-on-twitter

Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
100% 169M/169M [00:08<00:00, 22.1MB/s]



In [4]:
!unzip customer-support-on-twitter.zip

Archive:  customer-support-on-twitter.zip
  inflating: sample.csv              
  inflating: twcs/twcs.csv           


**Load Dataset**

In [5]:
!ls *.zip

customer-support-on-twitter.zip


In [6]:
import pandas as pd

df = pd.read_csv("twcs/twcs.csv")

In [7]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [8]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


**Explore the Dataset**

In [9]:
df.sample(5)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
2052258,2211220,BofA_Help,False,Thu Oct 19 14:49:28 +0000 2017,"@646305 Hi Matthew, please click below &amp; ...",NaN,2211221.0
1706672,1860809,132789,True,Mon Sep 18 10:42:12 +0000 2017,@187436 Great! Let us know how it turns out an...,1860808,1860810.0
186503,220158,AskLyft,False,Wed Oct 04 20:33:12 +0000 2017,@168521 Please report this to our Critical Res...,NaN,220159.0
2392683,2560847,comcastcares,False,Thu Nov 16 22:03:47 +0000 2017,"@726850 Alright, we're sorry to have made you ...",NaN,2560843.0
2417351,2586145,Ask_WellsFargo,False,Fri Nov 17 00:32:00 +0000 2017,@732796 Can you tell me more about what's goin...,2586146,2586147.0


In [10]:
print("Coulemns of dataset:\n",df.columns)

Coulemns of dataset:
 Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')


In [11]:
#show howmany companies tweets
df['inbound'].value_counts()

,count
inbound,
True,1537843
False,1273931


In [12]:
# show how many diff companies present
df[df['inbound']==False]['author_id'].nunique()

108

In [13]:
#companies name
brands_name = df[df['inbound']==False]['author_id'].value_counts()
brands_name.head(20)

,count
author_id,
AmazonHelp,169840
AppleSupport,106860
Uber_Support,56270
SpotifyCares,43265
Delta,42253
Tesco,38573
AmericanAir,36764
TMobileHelp,34317
comcastcares,33031


# **Create an AppleSupport Dataset**

**AppleSupport was selected for the following reasons:**

* It contains 106,860 support replies, providing a large and diverse dataset.
Customer issues are well-structured and fall into clear categories such as Apple ID, iCloud, device issues, software updates, battery, and payments.
* Apple's support responses are consistent, professional, and suitable for training and evaluating an AI support system.
* The dataset includes both routine and complex customer queries, making it suitable for implementing intent classification, AI response generation, and human escalation.
* The size and quality of the dataset are sufficient to build a reliable evaluation set while keeping the project manageable.

In [14]:
# Filter apple repies
Apple_Support_df = df[df['author_id'] == 'AppleSupport']
print("AppleSupport replies:", Apple_Support_df.shape)
Apple_Support_df.head()

AppleSupport replies: (106860, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
396,696,AppleSupport,False,Tue Oct 31 22:27:49 +0000 2017,@115854 We're here for you. Which version of t...,697,698.0
398,699,AppleSupport,False,Tue Oct 31 22:36:27 +0000 2017,@115854 Lets take a closer look into this issu...,NaN,697.0
401,701,AppleSupport,False,Tue Oct 31 22:26:49 +0000 2017,@115855 Let's go to DM for the next steps. DM ...,NaN,702.0
403,703,AppleSupport,False,Tue Oct 31 22:09:52 +0000 2017,@115855 Any steps tried since it started last ...,702,704.0
405,705,AppleSupport,False,Tue Oct 31 21:57:00 +0000 2017,@115855 That's great it has iOS 11.1 as we can...,"706,704",707.0


**customer ↔ AppleSupport conversation**

In [15]:
# create lookup table
Tweet_Lookup=df.set_index("tweet_id")
print(Tweet_Lookup)
print(Tweet_Lookup.shape)

           author_id  inbound                      created_at  \
tweet_id                                                        
1         sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
2             115712     True  Tue Oct 31 22:11:45 +0000 2017   
3             115712     True  Tue Oct 31 22:08:27 +0000 2017   
4         sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
5             115712     True  Tue Oct 31 21:49:35 +0000 2017   
...              ...      ...                             ...   
2987947   sprintcare    False  Wed Nov 22 08:43:51 +0000 2017   
2987948       823869     True  Wed Nov 22 08:35:16 +0000 2017   
2812240       121673     True  Thu Nov 23 04:13:07 +0000 2017   
2987949       AldiUK    False  Wed Nov 22 08:31:24 +0000 2017   
2987950       823870     True  Tue Nov 21 22:01:04 +0000 2017   

                                                       text response_tweet_id  \
tweet_id                                                                 

In [16]:
#Match each Apple reply with the customer tweet
conversation_pairs = []
for _, row in Apple_Support_df.iterrows():
    parent_id = row["in_response_to_tweet_id"]
    if pd.notna(parent_id):
        parent_id = int(parent_id)
        if parent_id in Tweet_Lookup.index:
            customer = Tweet_Lookup.loc[parent_id]
            conversation_pairs.append({
                "customer_tweet": customer["text"],
                "apple_reply": row["text"]
            })

In [17]:
# Convert to a DataFrame
conversation_df = pd.DataFrame(conversation_pairs)
print(conversation_df)

                                           customer_tweet  \
0                  @AppleSupport  https://t.co/NV0yucs0lB   
1       @AppleSupport The newest update. I️ made sure ...   
2       @AppleSupport Tried resetting my settings .. r...   
3       @AppleSupport This is what it looks like https...   
4       @AppleSupport I️ have an iPhone 7 Plus and yes...   
...                                                   ...   
106643  anyone having issues with #osx #HighSierra and...   
106644  Hey @AppleSupport - not being able to duplicat...   
106645  Yo @AppleSupport is that weird glitch w/ the c...   
106646  What the fuck @AppleSupport  my phone keeps ha...   
106647  Is anyone having #iphone issues with there pho...   

                                              apple_reply  
0       @115854 We're here for you. Which version of t...  
1       @115854 Lets take a closer look into this issu...  
2       @115855 Let's go to DM for the next steps. DM ...  
3       @115855 Any steps t

In [18]:
conversation_df.head()

,customer_tweet,apple_reply
0,@AppleSupport https://t.co/NV0yucs0lB,@115854 We're here for you. Which version of t...
1,@AppleSupport The newest update. I️ made sure ...,@115854 Lets take a closer look into this issu...
2,@AppleSupport Tried resetting my settings .. r...,@115855 Let's go to DM for the next steps. DM ...
3,@AppleSupport This is what it looks like https...,@115855 Any steps tried since it started last ...
4,@AppleSupport I️ have an iPhone 7 Plus and yes...,@115855 That's great it has iOS 11.1 as we can...


In [19]:
print("Pair:\n",conversation_df.shape)

Pair:
 (106648, 2)


In [20]:
# save the dataset
conversation_df.to_csv("apple_conversations.csv", index=False)

**Analyze Apple Customer Messages**

In [21]:
conversation_df.head(2)

,customer_tweet,apple_reply
0,@AppleSupport https://t.co/NV0yucs0lB,@115854 We're here for you. Which version of t...
1,@AppleSupport The newest update. I️ made sure ...,@115854 Lets take a closer look into this issu...


In [22]:
#Check dataset size
print(conversation_df.shape)

(106648, 2)


In [23]:
print("Missing value\n",conversation_df.isnull().sum())
print("Remove duplicate value:\n")
conversation_df = conversation_df.drop_duplicates()
print("Now Shape:",conversation_df.shape)

Missing value
 customer_tweet    0
apple_reply       0
dtype: int64
Remove duplicate value:

Now Shape: (106648, 2)


In [24]:
conversation_df.sample(20)

,customer_tweet,apple_reply
33802,@AppleSupport iPhone 7 across all email accoun...,@399401 What's the iOS version shown in Settin...
48632,Dear @115858 @118936 @AppleSupport when are th...,"@499244 We're happy to help. To start, are you..."
91459,@AppleSupport Help! I synced my phone and now ...,"@710365 Thanks for reaching out to us, we're h..."
6391,@AppleSupport It’s not just me. The phone is s...,@176838 Is the Messages app closing while you'...
32157,@AppleSupport Already done thrice.. this error...,"@389323 Nice work trying out the restart, and ..."
91511,@AppleSupport Screen freezing issue too,@710391 We'd like to see how we can help with ...
84873,@AppleSupport Ever since installing #ios11 I h...,@669596 Let's help to get that battery life wo...
22465,@115858 @AppleSupport I can’t believe how my i...,@311629 Let's work to get those apps working. ...
94943,@AppleSupport refuses to tell me what my son b...,@732341 We want to help. Do you see what was p...
44843,Where is @AppleSupport at with fixing this “I”...,@479831 We are here and happy to help! Go ahea...


In [25]:
# message length
conversation_df["tweet_length"] = conversation_df["customer_tweet"].str.len()
conversation_df["tweet_length"].describe()

,tweet_length
count,106648.000000
mean,109.261468
std,51.744364
min,7.000000
25%,72.000000
50%,108.000000
75%,138.000000
max,362.000000


In [26]:
conversation_df.sample(10)

,customer_tweet,apple_reply,tweet_length
43617,@AppleSupport @115858 @116333 I know this migh...,@223301 We're always on the lookout for custom...,163
15983,@116333 hello sir how are you its my pleasure ...,@257393 Go ahead and send us a DM so that we a...,278
6391,@AppleSupport It’s not just me. The phone is s...,@176838 Is the Messages app closing while you'...,131
84968,@AppleSupport so for one particular person I c...,@669784 We want to help you with Messages. We'...,141
18529,@AppleSupport 11.1.2 !,@276646 Thanks. Please tell us your current lo...,22
75446,@AppleSupport what the idea with my phone apps...,@622485 We'd be happy to look into this with y...,138
40940,@115858 Update:\n\nAlright @115858 I.T did I.T...,@456915 Let's see what we can find together. D...,78
98116,Bought a new @115858 iPhone from @115911 last ...,@293275 We're here to help! Congratulations on...,132
71091,@AppleSupport the latest software update ios11...,@598844 Having reliable battery life is import...,93
78757,@115858: PLEASE stop High Sierra from deciding...,@162456 Thanks for reaching out to us regardin...,277


# **Take a random sample**

In [27]:
golden_df = conversation_df.sample(n=200, random_state=42)
golden_df.head()

,customer_tweet,apple_reply,tweet_length
53139,@AppleSupport I️ can connect just fine but my ...,@516039 Understood. Check here under step 6: ...,150
89838,So @AppleSupport my speaker just gon stop work...,@273855 We'd like to help. Tell us what device...,85
5354,"@AppleSupport look, i just got this iPhone 8 a...",@170489 We want to make sure your iPhone is th...,119
105462,@814571 @AppleSupport please update the publis...,@814570 We are seeing todays date at this link...,236
7431,@AppleSupport recent update for IPad and IPhon...,@163247 Let us know which apps you're having t...,99


In [28]:
# Add new columns
golden_df["intent"] = ""
golden_df["auto_or_escalate"] = ""
golden_df["reason"] = ""

In [29]:
golden_df.to_csv("golden_dataset.csv", index=False)

In [30]:
from google.colab import files
files.download("golden_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# Define your intent keywords
intent_keywords = {
    "Apple ID/Login": [
        "apple id", "login", "log in", "password", "sign in",
        "account", "locked", "verification"
    ],

    "Battery": [
        "battery", "charge", "charging", "power",
        "drain", "low power"
    ],

    "iCloud": [
        "icloud", "backup", "photos", "storage", "sync"
    ],

    "App Store": [
        "app store", "itunes", "download", "install",
        "podcast", "music"
    ],

    "Payment/Billing": [
        "refund", "charged", "payment", "billing",
        "subscription", "purchase"
    ],

    "Software Update": [
        "ios", "update", "upgrade", "beta",
        "software", "version"
    ],

    "Device Issue": [
        "iphone", "ipad", "mac", "screen",
        "speaker", "camera", "touch",
        "freeze", "restart", "boot"
    ]
}

In [32]:
# Create a labeling function
def classify_intent(text):

    text = str(text).lower()

    for intent, keywords in intent_keywords.items():

        for keyword in keywords:

            if keyword in text:
                return intent

    return "Other"

In [33]:
golden_df["intent"] = golden_df["customer_tweet"].apply(classify_intent)

In [34]:
def escalation(intent):

    if intent in ["Payment/Billing"]:
        return "Escalate"

    return "Auto"

In [35]:
golden_df["auto_or_escalate"] = golden_df["intent"].apply(escalation)

In [36]:
reason_dict = {

    "Apple ID/Login":"Password reset or account guidance available.",

    "Battery":"Standard battery troubleshooting available.",

    "iCloud":"Sync and backup troubleshooting available.",

    "App Store":"Standard App Store troubleshooting available.",

    "Payment/Billing":"Billing issues require human review.",

    "Software Update":"Software troubleshooting available.",

    "Device Issue":"Device troubleshooting available.",

    "Other":"Needs further investigation."
}

golden_df["reason"] = golden_df["intent"].map(reason_dict)

In [37]:
golden_df.to_csv("golden_dataset_labeled.csv", index=False)

In [38]:
from google.colab import files
files.download("golden_dataset_labeled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>